# Executive Summary

**Objective:** 
To integrate geographical metadata, resolve string formatting inconsistencies, and engineer normalized metrics, transforming the raw data into a finalized state ready for exploratory data analysis.

**Data Flow:**
*   **Inputs:** 
    * `data\raw\internship_positions.parquet`
    * `data\external\administrative_divisions.parquet`
*   **Output:** `data\interim\internship_postings.parquet` (Exported to the interim directory to preserve datatypes)

**Key Operations Performed:**
1. **Data Integration (Province Mapping) [<u>[click]</u>](#1-data-integration):** Mapped raw job locations to their respective provinces by joining against the administrative divisions dictionary.
    * **String Cleaning [<u>[click]</u>](#11-string-cleaning):** Resolved mismatching geographic keys by standardizing text (lowercasing, stripping punctuation, and removing extraneous words).
    * **Manual Overrides [<u>[click]</u>](#12-manual-overrides):** Applied a manual dictionary mapping to achieve 100% province mapping with zero null values.
2. **Data Conversion [<u>[click]</u>](#2-data-conversion):** Cast `weekly_working_day` to a categorical datatype due to its low cardinality (2 unique values).
3. **Feature Engineering [<u>[click]</u>](#3-feature-engineering):**
    * Engineered the `acceptance_percentage` column (`100 * approved_quota / (1 + applicant_count)`).
    * Categorized `job_title` into a new `job_category` feature to reduce cardinality.
    * Binned all heavy right-skewed numericals (`requested_quota`, `approved_quota`, `applicant_count`, `acceptance_percentage`) to capture "whale" postings in an "Extreme" category without deleting them.
    * One-hot encoded `education_level` for downstream stakeholder consumption.
    * Extracted a new binary flag, `allows_all_majors`, by parsing the `job_description` column.
4. **Schema Finalization [<u>[click]</u>](#4-schema-finalization):** Reorganized the final 14 columns into a logical analytical structure before exporting.

# Setup & Imports

In [92]:
# Import libraries
import numpy as np
import pandas as pd

from src.config import EXTERNAL_DATA_DIR, INTERIM_DATA_DIR, RAW_DATA_DIR

In [93]:
# Load datasets
adm_divisions = pd.read_parquet(EXTERNAL_DATA_DIR / "administrative_divisions.parquet")
internship_positions = pd.read_parquet(RAW_DATA_DIR / "internship_positions.parquet")

# 1. Data Integration
Mapping raw job locations to their respective provinces by joining against the administrative divisions dictionary.

In [94]:
# Convert a regency-province table into a dictionary
adm_divisions_dict = dict(zip(adm_divisions["regency"], adm_divisions["province"]))


In [95]:
# Create a new column, `province`, by mapping with a dictionary
internship_positions["province"] = internship_positions["job_location"].map(adm_divisions_dict)
display(internship_positions.head())

,job_id,published_at,job_title,company,job_location,education_level,allowed_major,job_description,weekly_working_day,requested_quota,approved_quota,applicant_count,province
0,a240f2ba-12c0-4958-b416-c3e9c1d4e344,2026-07-16T12:58:55+07:00,PSIKOLOG,RUMAH TAHANAN NEGARA KELAS IIB SIBUHUAN,Kab. Padang Lawas,Profession,Psikologi,1. Melakukan asesmen psikologis terhadap anak ...,6,1,1,0,Sumatera Utara
1,a240f336-b6b7-47fe-8096-5b4c2eb2ed1f,2026-07-16T12:58:55+07:00,PSIKIATER,RUMAH TAHANAN NEGARA KELAS IIB SIBUHUAN,Kab. Padang Lawas,Profession,Kedokteran,1. Menangani gangguan kesehatan jiwa warga bin...,6,1,1,0,Sumatera Utara
2,a242e6a4-2c7a-4ce4-8a57-7aff601a5e2c,2026-07-16T12:57:40+07:00,PSIKIATER,RUMAH TAHANAN NEGARA KELAS IIB SALATIGA,Kota Salatiga,Profession,Kedokteran,1. Menangani gangguan kesehatan jiwa warga bin...,5,1,1,0,Jawa Tengah
3,a24138aa-bcf6-4940-bfaf-dfe5dbf66ca6,2026-07-16T12:50:42+07:00,PERAWAT KESEHATAN,LEMBAGA PEMASYARAKATAN KELAS III SUKAMARA,Kab. Sukamara,Bachelor,Ilmu Gizi,1. Memberikan perawatan kesehatan umum dan tin...,6,1,1,0,Kalimantan Tengah
4,a23f7c52-9123-46e9-bb5e-be376b1d77f2,2026-07-16T12:38:52+07:00,Psikiater,LEMBAGA PEMASYARAKATAN KELAS III ARJASA,Kab. Sumenep,Bachelor,Kedokteran,1. Menangani gangguan kesehatan jiwa warga bin...,6,1,1,0,Jawa Timur


In [96]:
# Find regencies from `internship_positions` that have no match 
# with their corresponding provinces
display(
    internship_positions[
        internship_positions["province"].isnull()
    ].groupby("job_location")["job_id"].count().sort_values(ascending=False)
)

job_location
Kota Batam                          221
Kota Palangkaraya                   120
Kota Dumai                           77
Kab. Pangkajene Kepulauan            57
Kota Bau Bau                         56
Kota Banjarbaru                      48
Kab. Siak                            46
Kota Lubuk Linggau                   43
Kab. Gunungkidul                     42
Kab. Karangasem                      39
Kota Sawahlunto                      38
Kab. Batanghari                      31
Kab. Tulang Bawang                   28
Kota Pematangsiantar                 25
Unknown Location                     24
Kab. Kotabaru                        24
Kab. Banyuasin                       23
Kab. Labuhanbatu                     23
Kota Pare Pare                       20
Kab. Tojo Una Una                    18
Kab. Pahuwato                        17
Kab. Kep. Siau Tagulandang Biaro     17
Kepulauan Tanimbar                   15
Kab Timor Tengah Selatan             15
Kab. Toli Toli             

In [97]:
# Get all regencies from `adm_divisions` that have no match
# with job locations from `internship_positions`
matched_regencies = list(set(internship_positions[
    ~internship_positions.province.isnull()
]["job_location"].to_list()))

unmatched_regencies = list(set(adm_divisions.regency.to_list()) - set(matched_regencies))

display(
    adm_divisions[
        adm_divisions.regency.isin(unmatched_regencies)
    ][["regency", "province"]].sort_values("regency", ascending=False)
)

,regency,province
70,Kota Sawah Lunto,Sumatera Barat
50,Kota Pematang Siantar,Sumatera Utara
420,Kota Parepare,Sulawesi Selatan
341,Kota Palangka Raya,Kalimantan Tengah
114,Kota Lubuklinggau,Sumatera Selatan
...,...,...
90,Kab. Batang Hari,Jambi
104,Kab. Banyu Asin,Sumatera Selatan
144,Kab. Bangka Selatan,Kepulauan Bangka Belitung
385,Kab. Banggai Kepulauan,Sulawesi Tengah


## 1.1 String Cleaning

In [98]:
# Fallback mapping: Stripping punctuation and whitespace resolves mismatches
# caused by inconsistent data entry on the scraped platform.
plain_adm_div_dict = dict(
    zip(
        adm_divisions["regency"]
        .str.lower()
        .str.replace(r"\sdan\s", " ", case=False, regex=True)
        .str.replace(r"\W", "", regex=True),
        adm_divisions["province"],
    )
)

display(plain_adm_div_dict)

mask = internship_positions["province"].isnull()
internship_positions.loc[mask, "province"] = (
    internship_positions.loc[mask, "job_location"]
    .str.lower()
    .str.replace(r"\sdan\s", " ", case=False, regex=True)
    .str.replace(r"\W", "", regex=True)
    .map(plain_adm_div_dict)
)

{'kabsimeulue': 'Aceh',
 'kabacehsingkil': 'Aceh',
 'kabacehselatan': 'Aceh',
 'kabacehtenggara': 'Aceh',
 'kabacehtimur': 'Aceh',
 'kabacehtengah': 'Aceh',
 'kabacehbarat': 'Aceh',
 'kabacehbesar': 'Aceh',
 'kabpidie': 'Aceh',
 'kabbireuen': 'Aceh',
 'kabacehutara': 'Aceh',
 'kabacehbaratdaya': 'Aceh',
 'kabgayolues': 'Aceh',
 'kabacehtamiang': 'Aceh',
 'kabnaganraya': 'Aceh',
 'kabacehjaya': 'Aceh',
 'kabbenermeriah': 'Aceh',
 'kabpidiejaya': 'Aceh',
 'kotabandaaceh': 'Aceh',
 'kotasabang': 'Aceh',
 'kotalangsa': 'Aceh',
 'kotalhokseumawe': 'Aceh',
 'kotasubulussalam': 'Aceh',
 'kabnias': 'Sumatera Utara',
 'kabmandailingnatal': 'Sumatera Utara',
 'kabtapanuliselatan': 'Sumatera Utara',
 'kabtapanulitengah': 'Sumatera Utara',
 'kabtapanuliutara': 'Sumatera Utara',
 'kabtobasamosir': 'Sumatera Utara',
 'kablabuhanbatu': 'Sumatera Utara',
 'kabasahan': 'Sumatera Utara',
 'kabsimalungun': 'Sumatera Utara',
 'kabdairi': 'Sumatera Utara',
 'kabkaro': 'Sumatera Utara',
 'kabdeliserdang': '

In [99]:
# Check for the missing values again by finding regencies from `internship_positions`
# that have no match with their corresponding provinces
display(
    internship_positions[
        internship_positions["province"].isnull()
    ].groupby("job_location")["job_id"].count().sort_values(ascending=False)
)

job_location
Unknown Location                    24
Kab. Kep. Siau Tagulandang Biaro    17
Kab. Pahuwato                       17
Kepulauan Tanimbar                  15
Kab. Mahakam Ulu                     1
Name: job_id, dtype: int64

## 1.2 Manual Overrides

In [100]:
# Manual overrides for edge cases missing from the standard division dataset
manual_dict = {
    "Kab. Kep. Siau Tagulandang Biaro": "Sulawesi Utara",
    "Kab. Mahakam Ulu": "Kalimantan Timur",
    "Kab. Pahuwato": "Gorontalo",
    "Kepulauan Tanimbar": "Maluku",
    "Unknown Location": "Unknown Location",
}

mask = internship_positions["province"].isnull()
internship_positions.loc[mask, "province"] = internship_positions.loc[
    mask, "job_location"
].map(manual_dict)

In [101]:
# Check for the missing values again by finding regencies from `internship_positions`
# that have no match with their corresponding provinces
display(
    internship_positions[
        internship_positions["province"].isnull()
    ].groupby("job_location")["job_id"].count().sort_values(ascending=False)
)

Series([], Name: job_id, dtype: int64)

In [102]:
# Rename Column `job_location` to `regency_city`
internship_positions.rename(columns={"job_location": "regency_city"}, inplace=True)

In [103]:
# Fix some regency and city names
internship_positions["regency_city"] = (
    internship_positions.regency_city
    .str.replace(r"^Kab\s", r"Kab. ", regex=True)
    .str.replace("Pahuwato", "Pohuwato")
    .str.replace(r"^Kepulauan\s", r"Kab. Kep. ", regex=True)
    .str.replace(r"\sKepulauan\s", r" Kep. ", regex=True)
)

# 2. Data Conversion
Casting `weekly_working_day` to a categorical datatype due to its low cardinality (2 unique values).

In [104]:
# Cast the data type of `weekly_working_day` to string
internship_positions = internship_positions.astype({"weekly_working_day": "str"})

internship_positions.info()

<class 'pandas.DataFrame'>
RangeIndex: 28322 entries, 0 to 28321
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   job_id              28322 non-null  str  
 1   published_at        28322 non-null  str  
 2   job_title           28322 non-null  str  
 3   company             28322 non-null  str  
 4   regency_city        28322 non-null  str  
 5   education_level     28322 non-null  str  
 6   allowed_major       28322 non-null  str  
 7   job_description     28322 non-null  str  
 8   weekly_working_day  28322 non-null  str  
 9   requested_quota     28322 non-null  int64
 10  approved_quota      28322 non-null  int64
 11  applicant_count     28322 non-null  int64
 12  province            28322 non-null  str  
dtypes: int64(3), str(10)
memory usage: 20.0 MB


# 3. Feature Engineering

## 3.1 Feature Construction
Constructing the `acceptance_percentage` column (`100 * approved_quota / (1 + applicant_count)`)

In [105]:
# Create Column `acceptance_percentage`
internship_positions["acceptance_percentage"] = round(
    100
    * internship_positions["approved_quota"]
    / (internship_positions["applicant_count"].add(1)),
    2,
)

display(internship_positions.sample(10))

,job_id,published_at,job_title,company,regency_city,education_level,allowed_major,job_description,weekly_working_day,requested_quota,approved_quota,applicant_count,province,acceptance_percentage
11847,a23dee79-f25d-4785-a8c8-a206ddad5d96,2026-07-16T12:06:45+07:00,Sekretaris,BPS Kabupaten Aceh Singkil,Kota Adm. Jakarta Pusat,"Bachelor, Diploma","Administrasi, Manajemen Administrasi, Sekretar...","Mengelola administrasi perkantoran, penjadwala...",5,4,4,25,DKI Jakarta,15.38
25528,a2440b18-1072-4ee7-95fc-53aebc43136d,2026-07-16T12:10:27+07:00,PRANATA HUBUNGAN MASYARAKAT,KANTOR WILAYAH DIREKTORAT JENDERAL IMIGRASI SU...,Kota Padang,Bachelor,"Informasi dan Humas, Ilmu Hubungan Masyarakat,...",Melaksanakan kegiatan pengelolaan informasi da...,5,2,2,35,Sumatera Barat,5.56
9952,a2415b0d-e469-47fd-b730-51eabc4490dc,2026-07-16T12:06:14+07:00,Staf Komunikasi Dan Kesekretariatan,BPJS Kesehatan Kantor Cabang Sampit,Kab. Kotawaringin Timur,"Diploma, Bachelor","Komunikasi, Jurnalistik, Ilmu Komunikasi, Desa...",Membantu melaksanakan kegiatan administratif d...,5,1,1,6,Kalimantan Tengah,14.29
26745,a243d731-a72b-45c8-8e6f-2cd2b2612eba,2026-07-16T12:43:46+07:00,Pranata Laboratorium Kesehatan,Rumah Sakit Umum Pusat Dr. Wahidin Sudirohusod...,Kota Makassar,Diploma,Teknologi Laboratorium Medis,Melaksanakan pemeriksaan laboratorium mulai da...,5,4,4,84,Sulawesi Selatan,4.71
4966,a2391827-a8a7-4e69-ba78-fcaa3b29dd5f,2026-07-16T10:09:12+07:00,Perawat Pelaksana,PT Thamrin Sinar Surya,Kota Medan,"Diploma, Profession","Keperawatan, Ilmu Keperawatan, Keperawatan Gigi",1.\tPelatihan keperawatan (uraian tugas ),6,30,30,126,Sumatera Utara,23.62
1638,a24146dc-a815-4820-9f71-3fbb25d93ca9,2026-07-16T10:44:41+07:00,Production,PT. Haeng Nam Sejahtera Indonesia,Kab. Bogor,"Diploma, Bachelor, Profession","Teknik Industri, Teknik Mesin, Teknik Manufak...",Memahami proses bisnis perusahaan manufaktur s...,6,15,15,39,Jawa Barat,37.50
14288,a2435d30-2469-43e2-8801-fc1e1a0773d6,2026-07-16T12:10:11+07:00,DNEK - Asisten Analis Penilaian Emiten dan Pe...,OJKI,Kota Adm. Jakarta Pusat,Bachelor,"Keuangan, Manajemen, Ekonomi, Akuntansi","1. Memahami ketentuan, peraturan, dan perundan...",5,2,2,14,DKI Jakarta,13.33
522,a238f3b0-3e5a-4c63-91d1-5566042833b3,2026-07-16T10:31:28+07:00,QC Produksi - PT. ESGI Klego,PT Eco Smart Garment Indonesia,Kab. Boyolali,"Diploma, Bachelor","Tata Busana, Pendidikan Tata Busana, Teknik In...",$25,5,7,7,13,Jawa Tengah,50.00
1948,a242ad39-9bf5-4bfd-a1f5-f8cb88b23b3b,2026-07-16T12:34:37+07:00,Asisten Pengelola Keuangan,BPS Kabupaten Kepulauan Tanimbar,Kab. Kep. Tanimbar,"Bachelor, Diploma","Manajemen Keuangan, Perpajakan, Ekonomi, Akunt...",Membantu proses pengelolaan administrasi keuan...,5,1,1,3,Maluku,25.00
10447,a2273d1c-bd17-4001-8700-ea9bea7284c5,2026-07-16T10:47:16+07:00,"Technician, Electrical Maintenance",PT. Indah Kiat Pulp And Paper Tbk,Kab. Serang,"Diploma, Bachelor","Pendidikan Vokasional Teknik Elektronika, Tekn...","1. Knowledge of high voltage cables (20KV, 3,3...",5,1,1,6,Banten,14.29


## 3.2 Feature Transformation
### 3.2.1 Text Classification
Categorizing `job_title` into a new `job_category` feature to reduce cardinality.

In [106]:
# Create a custom function to categorize the jobs
def categorize_job(title):
    if pd.isna(title):
        return "Uncategorized"
    
    t = str(title).lower()
    
    # 1. Healthcare & Medical (Added severe typos, hospital codes, and specialized terms)
    if any(w in t for w in ["perawat", "ners", "nurse", "psikiat", "spikiat", "psikat", "pskiat", "psikolog", "piskolog", "pskolog", "medis", "medic", "medik", "gizi", "nutri", "diet", "dokter", "doker", "doktor", "bidan", "apotek", "aptoker", "farmasi", "pharmac", "fisio", "physio", "radio", "sanitari", "sanitasi", "kesehatan", "promkes", "okupasi", "elektromedis", "atem", "terapi", "therap", "klinik", "atlm", "epidemiolog", "anestesi", "anastesi", "rekam medi", "perekam", "mr ", "casemix", "coder", "koder", "cssd", "ipsrs", "audiolog", "orthotic", "mcu", "ranap", "igd", "poliklinik", "vk ", "bersalin", "hemodialisa", "kardiovaskuler", "cardiovascular", "refraksi", "kebidanan", "keperawatan", "patologi", "mikrobiologi", "imunologi", "darah", "ambul", "hospital", "rehabilitasi", "admission"]):
        return "Healthcare & Medical"
    
    # 2. IT & Data
    elif any(w in t for w in ["komputer", "it ", " it", "programmer", "developer", "software", "data", "sistem", "system", "ui/", "/ux", "network", "cyber", "aplikasi", "application", "backend", "frontend", "website", "web", "informatika", "pusdatin", "jaringan", "ai ", "machine learning", "bda", "digital", "erp", "sap ", "helpdesk", "support it", "noc ", "rpa ", "command center", "dashboard", "cloud", "iot", "analytic"]):
        return "IT & Data"
        
    # 3. Engineering & Maintenance
    elif any(w in t for w in ["teknis", "technician", "maintenance", "engineer", "mekanik", "mechanic", "drafter", "drawing", "listrik", "sipil", "civil", "bangunan", "hvac", "otomotif", "mesin", "machine", "welder", "welding", "proyek", "project", "elektro", "electric", "arsitek", "architect", "maint", "equipment", "facility", "sarana", "prasarana", "geologi", "tambang", "mining", "instrument", "surveyor", "craft", "plumbing", "geofisika", "seismik", "geodesi", "geomatika", "toolman", "inspector", "inspektur", "konstruksi", "construction", "pipa", "baja", "otomasi", "automation"]):
        return "Engineering & Maintenance"
        
    # 4. Manufacturing, QA & Production
    elif any(w in t for w in ["produksi", "production", "operator", "qc", "qa", "quality", "pabrik", "manufacturing", "assembly", "packaging", "mutu", "plant", "molding", "mould", "mold", "pattern maker", "slitting", "blown film", "sewing", "garment", "textile", "printing", "finishing", "curing", "mixing", "extruder", "ppic", "rnd", "r&d", "research", "reserch", "set up", "cleanning", "rewinding", "laminasi", "improvement", "pdca", "koe ", "lean", "mill", "helper", "pe ", "ie "]):
        return "Manufacturing, QA & Production"
        
    # 5. Finance & Banking
    elif any(w in t for w in ["keuangan", "akuntansi", "accounting", "akuntan", "pajak", "tax", "auditor", "audit", "bendahara", "anggaran", "finance", "treasury", "billing", "kasir", "cashier", "credit", "kredit", "loan", "pembiayaan", "funding", "transaction", "collection", "receivable", "payable", "wealth", "insurance", "asuransi", "actuary", "aktuaria", "bank", "bni", "teller", "pawning", "micro", "invest", "budget", "cost "]):
        return "Finance & Banking"
        
    # 6. Sales, Marketing & Hospitality
    elif any(w in t for w in ["barista", "cook", "koki", "pastry", "bakery", "culinary", "chef", "layanan", "frontliner", "frontlner", "sales", "marketing", "f&b", "fb ", "store", "customer", "receptionist", "pemasaran", "pramusaji", "hotel", "event", "reservation", "guest", "hospitality", "dancer", "entertainment", "commercial", "merchandis", "retail", "promot", "promosi", "brand", "business development", "bd ", "partnership", "account executive", "masak", "food", "beverage", "catering", "kitchen", "tour ", "travel", "room", "bro", "activation"]):
        return "Sales, Marketing & Hospitality"
        
    # 7. Media, PR & Creative
    elif any(w in t for w in ["humas", "kehumasan", "design", "desain", "kreatif", "creative", "video", "animator", "animation", "content", "konten", "sosial media", "social media", "sosmed", "kol ", "publisitas", "jurnalis", "journalist", "wartawan", "editor", "multimedia", "reporter", "fotografer", "photographer", "broadcasting", "komunikasi", "communication", "visual", "vm artist", "motion", "copywriter", "writer", "art ", "talent", "audio", "camera", "campaign", "publikasi", "publik", "illustrator", "media", "broadcast", "creator"]):
        return "Media, PR & Creative"
        
    # 8. Legal, Risk & Compliance
    elif any(w in t for w in ["hukum", "legal", "law", "compliance", "kepatuhan", "risk", "risiko", "hse", "hsse", "qhse", "she", "ehs", "safety", "k3", "security", "keamanan", "fraud", "investigasi", "pengaduan", "maladministrasi", "litigasi", "regulas", "regulatory", "sertifikasi", "perizinan", "izin", "kekayaan intelektual"]):
        return "Legal, Risk & Compliance"
        
    # 9. Logistics & Supply Chain
    elif any(w in t for w in ["gudang", "warehouse", "logistik", "logistic", "exim", "supply chain", "scm", "inventory", "pengadaan", "purchasing", "procurement", "buyer", "ekspor", "impor", "export", "import", "cargo", "shipping", "freight", "delivery", "transport", "fleet", "ekspeditor", "harbour", "port ", "bandara", "airport", "pelabuhan", "aviation", "aero", "aircraft", "checker", "terminal"]):
        return "Logistics & Supply Chain"
        
    # 10. Education, Training & Government
    elif any(w in t for w in ["kebijakan", "pemerintahan", "penelaah", "pengawas", "asn", "biro", "kementerian", "pemda", "pns", "diplomat", "instruktur", "pelatihan", "pembelajaran", "tentor", "pengajar", "edukator", "diklat", "akademik", "guru", "dosen", "widyaiswara", "pusat", "badan", "tutor", "statistik", "statistisi", "peneliti", "pustaka", "kearsipan", "arsip", "kurator", "laporan", "pelaporan", "penyusun", "evaluasi", "pengolah", "dokumen", "evaluator", "pemeriksaan"]):
        return "Education, Training & Government"
        
    # 11. Agriculture & Environment
    elif any(w in t for w in ["pertanian", "perikanan", "peternakan", "perkebunan", "agribisnis", "kehutanan", "lingkungan", "agronomi", "pangan", "tambak", "tanaman", "kebun", "hewan", "hutan", "forestry", "environment", "sustainability", "esg", "limbah", "waste", "marine", "hydro", "iklim", "climate", "budidaya", "ternak", "nelayan", "satwa", "flora", "fauna", "ekologi", "air "]):
        return "Agriculture & Environment"
        
    # 12. Language & Translation
    elif any(w in t for w in ["isyarat", "penerjemah", "translator", "interpreter", "language", "mandarin", "japanese", "english", "bahasa"]):
        return "Language & Translation"
        
    # 13. Correctional & Social Services
    elif any(w in t for w in ["pembinaan", "kepribadian", "pembimbing kemasyarakatan", "warga binaan", "kegiatan kerja", "rohani", "sosial", "pemasyarakatan", "community", "csr", "tjsl", "bina", "klien", "konselor"]):
        return "Correctional & Social Services"

    # 14. HR, Admin & Management (General catch-alls placed at the very end)
    elif any(w in t for w in ["sdm", "human resource", "hr", "ga", "general affair", "administrasi", "admin", "bmn", "sekretaris", "secretary", "tata usaha", "tu ", "personil", "personalia", "rekrutmen", "recruitment", "talent acquisition", "od ", "organization development", "umum", "fasilitas", "manajemen", "management", "manager", "pmo", "strategi", "koordinator", "coordinator", "supervisor", "spv", "director", "operasional", "operation", "asset", "aset", "clerical", "sarana", "pejabat", "pengelola", "asisten", "officer", "staff", "staf", "pelaksana", "magang", "intern", "consultant", "konsultan", "planner"]):
        return "HR, Admin & Management"
        
    else:
        return "Other"

In [107]:
# Apply the function to create the new column
internship_positions["job_category"] = internship_positions["job_title"].apply(categorize_job)

# Verify the distribution of the new categories
display(internship_positions["job_category"].value_counts())

job_category
HR, Admin & Management              5176
Healthcare & Medical                4063
IT & Data                           2802
Media, PR & Creative                2570
Sales, Marketing & Hospitality      2150
Correctional & Social Services      2005
Finance & Banking                   1707
Engineering & Maintenance           1701
Education, Training & Government    1294
Manufacturing, QA & Production      1083
Legal, Risk & Compliance            1036
Other                                994
Logistics & Supply Chain             837
Agriculture & Environment            597
Language & Translation               307
Name: count, dtype: int64

### 3.2.2 Binning (Discretization)
Binning all heavy right-skewed numericals (`requested_quota`, `approved_quota`, `applicant_count`, `acceptance_percentage`) to capture "whale" postings in an "Extreme" category without deleting them.

In [110]:
# Bin `requested_quota` and `approved_quota`
quota_edges = [1, 2, 10, 50, np.inf]
quota_labels = ["1 to 2", "3 to 10", "11 to 50", "50+"]

internship_positions["requested_quota_category"] = pd.cut(
    internship_positions["requested_quota"],
    bins=quota_edges,
    labels=quota_labels,
    include_lowest=True
)

internship_positions["approved_quota_category"] = pd.cut(
    internship_positions["approved_quota"],
    bins=quota_edges,
    labels=quota_labels,
    include_lowest=True
)

display(internship_positions.head())

,job_id,published_at,job_title,company,regency_city,education_level,allowed_major,job_description,weekly_working_day,requested_quota,approved_quota,applicant_count,province,acceptance_percentage,job_category,requested_quota_category,approved_quota_category
0,a240f2ba-12c0-4958-b416-c3e9c1d4e344,2026-07-16T12:58:55+07:00,PSIKOLOG,RUMAH TAHANAN NEGARA KELAS IIB SIBUHUAN,Kab. Padang Lawas,Profession,Psikologi,1. Melakukan asesmen psikologis terhadap anak ...,6,1,1,0,Sumatera Utara,100.0,Healthcare & Medical,1 to 2,1 to 2
1,a240f336-b6b7-47fe-8096-5b4c2eb2ed1f,2026-07-16T12:58:55+07:00,PSIKIATER,RUMAH TAHANAN NEGARA KELAS IIB SIBUHUAN,Kab. Padang Lawas,Profession,Kedokteran,1. Menangani gangguan kesehatan jiwa warga bin...,6,1,1,0,Sumatera Utara,100.0,Healthcare & Medical,1 to 2,1 to 2
2,a242e6a4-2c7a-4ce4-8a57-7aff601a5e2c,2026-07-16T12:57:40+07:00,PSIKIATER,RUMAH TAHANAN NEGARA KELAS IIB SALATIGA,Kota Salatiga,Profession,Kedokteran,1. Menangani gangguan kesehatan jiwa warga bin...,5,1,1,0,Jawa Tengah,100.0,Healthcare & Medical,1 to 2,1 to 2
3,a24138aa-bcf6-4940-bfaf-dfe5dbf66ca6,2026-07-16T12:50:42+07:00,PERAWAT KESEHATAN,LEMBAGA PEMASYARAKATAN KELAS III SUKAMARA,Kab. Sukamara,Bachelor,Ilmu Gizi,1. Memberikan perawatan kesehatan umum dan tin...,6,1,1,0,Kalimantan Tengah,100.0,Healthcare & Medical,1 to 2,1 to 2
4,a23f7c52-9123-46e9-bb5e-be376b1d77f2,2026-07-16T12:38:52+07:00,Psikiater,LEMBAGA PEMASYARAKATAN KELAS III ARJASA,Kab. Sumenep,Bachelor,Kedokteran,1. Menangani gangguan kesehatan jiwa warga bin...,6,1,1,0,Jawa Timur,100.0,Healthcare & Medical,1 to 2,1 to 2


In [111]:
# Bin Column `applicant_count`
applicant_edges = [0, 5, 10, 20, 50, np.inf]
applicant_labels = ["0 to 5", "6 to 10", "11 to 20", "21 to 50", "50+"]

internship_positions["applicant_count_category"] = pd.cut(
    internship_positions["applicant_count"],
    bins=applicant_edges,
    labels=applicant_labels,
    include_lowest=True
)

display(internship_positions.head())

,job_id,published_at,job_title,company,regency_city,education_level,allowed_major,job_description,weekly_working_day,requested_quota,approved_quota,applicant_count,province,acceptance_percentage,job_category,requested_quota_category,approved_quota_category,applicant_count_category
0,a240f2ba-12c0-4958-b416-c3e9c1d4e344,2026-07-16T12:58:55+07:00,PSIKOLOG,RUMAH TAHANAN NEGARA KELAS IIB SIBUHUAN,Kab. Padang Lawas,Profession,Psikologi,1. Melakukan asesmen psikologis terhadap anak ...,6,1,1,0,Sumatera Utara,100.0,Healthcare & Medical,1 to 2,1 to 2,0 to 5
1,a240f336-b6b7-47fe-8096-5b4c2eb2ed1f,2026-07-16T12:58:55+07:00,PSIKIATER,RUMAH TAHANAN NEGARA KELAS IIB SIBUHUAN,Kab. Padang Lawas,Profession,Kedokteran,1. Menangani gangguan kesehatan jiwa warga bin...,6,1,1,0,Sumatera Utara,100.0,Healthcare & Medical,1 to 2,1 to 2,0 to 5
2,a242e6a4-2c7a-4ce4-8a57-7aff601a5e2c,2026-07-16T12:57:40+07:00,PSIKIATER,RUMAH TAHANAN NEGARA KELAS IIB SALATIGA,Kota Salatiga,Profession,Kedokteran,1. Menangani gangguan kesehatan jiwa warga bin...,5,1,1,0,Jawa Tengah,100.0,Healthcare & Medical,1 to 2,1 to 2,0 to 5
3,a24138aa-bcf6-4940-bfaf-dfe5dbf66ca6,2026-07-16T12:50:42+07:00,PERAWAT KESEHATAN,LEMBAGA PEMASYARAKATAN KELAS III SUKAMARA,Kab. Sukamara,Bachelor,Ilmu Gizi,1. Memberikan perawatan kesehatan umum dan tin...,6,1,1,0,Kalimantan Tengah,100.0,Healthcare & Medical,1 to 2,1 to 2,0 to 5
4,a23f7c52-9123-46e9-bb5e-be376b1d77f2,2026-07-16T12:38:52+07:00,Psikiater,LEMBAGA PEMASYARAKATAN KELAS III ARJASA,Kab. Sumenep,Bachelor,Kedokteran,1. Menangani gangguan kesehatan jiwa warga bin...,6,1,1,0,Jawa Timur,100.0,Healthcare & Medical,1 to 2,1 to 2,0 to 5


In [112]:
# Bin Column `acceptance_percentage`
acceptance_edges = [0, 10, 25, 50, np.inf]
acceptance_labels = ["0 - 10%", "11 - 25%", "26 - 50%", "50%+"]

internship_positions["acceptance_percentage_category"] = pd.cut(
    internship_positions["acceptance_percentage"],
    bins=acceptance_edges,
    labels=acceptance_labels,
    include_lowest=True
)

display(internship_positions.head())

,job_id,published_at,job_title,company,regency_city,education_level,allowed_major,job_description,weekly_working_day,requested_quota,approved_quota,applicant_count,province,acceptance_percentage,job_category,requested_quota_category,approved_quota_category,applicant_count_category,acceptance_percentage_category
0,a240f2ba-12c0-4958-b416-c3e9c1d4e344,2026-07-16T12:58:55+07:00,PSIKOLOG,RUMAH TAHANAN NEGARA KELAS IIB SIBUHUAN,Kab. Padang Lawas,Profession,Psikologi,1. Melakukan asesmen psikologis terhadap anak ...,6,1,1,0,Sumatera Utara,100.0,Healthcare & Medical,1 to 2,1 to 2,0 to 5,50%+
1,a240f336-b6b7-47fe-8096-5b4c2eb2ed1f,2026-07-16T12:58:55+07:00,PSIKIATER,RUMAH TAHANAN NEGARA KELAS IIB SIBUHUAN,Kab. Padang Lawas,Profession,Kedokteran,1. Menangani gangguan kesehatan jiwa warga bin...,6,1,1,0,Sumatera Utara,100.0,Healthcare & Medical,1 to 2,1 to 2,0 to 5,50%+
2,a242e6a4-2c7a-4ce4-8a57-7aff601a5e2c,2026-07-16T12:57:40+07:00,PSIKIATER,RUMAH TAHANAN NEGARA KELAS IIB SALATIGA,Kota Salatiga,Profession,Kedokteran,1. Menangani gangguan kesehatan jiwa warga bin...,5,1,1,0,Jawa Tengah,100.0,Healthcare & Medical,1 to 2,1 to 2,0 to 5,50%+
3,a24138aa-bcf6-4940-bfaf-dfe5dbf66ca6,2026-07-16T12:50:42+07:00,PERAWAT KESEHATAN,LEMBAGA PEMASYARAKATAN KELAS III SUKAMARA,Kab. Sukamara,Bachelor,Ilmu Gizi,1. Memberikan perawatan kesehatan umum dan tin...,6,1,1,0,Kalimantan Tengah,100.0,Healthcare & Medical,1 to 2,1 to 2,0 to 5,50%+
4,a23f7c52-9123-46e9-bb5e-be376b1d77f2,2026-07-16T12:38:52+07:00,Psikiater,LEMBAGA PEMASYARAKATAN KELAS III ARJASA,Kab. Sumenep,Bachelor,Kedokteran,1. Menangani gangguan kesehatan jiwa warga bin...,6,1,1,0,Jawa Timur,100.0,Healthcare & Medical,1 to 2,1 to 2,0 to 5,50%+


## 3.3 Feature Encoding
One-hot encoding `education_level` for downstream stakeholder consumption.

In [113]:
# One hot encode `education_level`
ed_level_dummies = internship_positions["education_level"].str.lower().str.get_dummies(sep=", ")
ed_level_dummies = ed_level_dummies.replace({0: "No", 1: "Yes"}).add_prefix("allows_")
ed_level_dummies = ed_level_dummies.add_suffix("_level")

internship_positions = pd.concat([internship_positions, ed_level_dummies], axis=1)
display(internship_positions.sample(5))

,job_id,published_at,job_title,company,regency_city,education_level,allowed_major,job_description,weekly_working_day,requested_quota,...,province,acceptance_percentage,job_category,requested_quota_category,approved_quota_category,applicant_count_category,acceptance_percentage_category,allows_bachelor_level,allows_diploma_level,allows_profession_level
8739,a2441e89-74da-4b84-9c41-061e0378b531,2026-07-16T11:50:40+07:00,Digital Content Creator,BPVP Pangkep,Kab. Pangkajene Kepulauan,"Diploma, Bachelor","Penyiaran, Multimedia, Desain Komunikasi Visua...","Memproduksi aset publikasi (foto/video), melak...",5,2,...,Sulawesi Selatan,16.67,IT & Data,1 to 2,1 to 2,11 to 20,11 - 25%,Yes,Yes,No
22182,a2401e5b-62e7-46d9-900b-14cff543a68c,2026-07-16T10:35:17+07:00,Content Creator Intern,PT Fore Kopi Indonesia,Kota Adm. Jakarta Pusat,Bachelor,"Multimedia, Ilmu Komunikasi, Komunikasi Digita...","1. Create engaging content for TikTok, Instagr...",5,2,...,DKI Jakarta,8.00,"Media, PR & Creative",1 to 2,1 to 2,21 to 50,0 - 10%,Yes,No,No
4958,a238221e-78e3-40c7-bd40-1333214c8497,2026-07-16T10:24:07+07:00,Loyalty Program Intern,PT Citilink Indonesia,Kota Tangerang,Bachelor,"Teknologi Informasi, Manajemen, Teknik Industr...",1. Membantu memastikan penanganan operasional ...,5,5,...,Banten,22.73,"HR, Admin & Management",3 to 10,3 to 10,21 to 50,11 - 25%,Yes,No,No
5022,a240d770-7e09-4fef-b53f-ac9c347aafac,2026-07-16T12:59:31+07:00,PENGELOLA KEHUMASAN DAN SDM,BALAI PEMASYARAKATAN KELAS II MERAUKE,Kab. Merauke,Bachelor,"Kearsipan, Teknik informatika, sistem informas...",1. Menyusun materi layanan informasi untuk med...,5,3,...,Papua,21.43,"Media, PR & Creative",3 to 10,3 to 10,11 to 20,11 - 25%,Yes,No,No
2122,a23f8e99-d2d5-4376-b064-60e182bd316c,2026-07-16T12:10:19+07:00,PENGELOLA KEGIATAN KERJA,LEMBAGA PEMASYARAKATAN KELAS IIB MUARA DUA,Kab. Ogan Komering Ulu Selatan,Bachelor,"Pengelolaan Agribisnis Perkebunan, Budidaya Ta...",1.\tMenyusun rencana dan mengelola kegiatan ke...,6,1,...,Sumatera Selatan,25.00,Correctional & Social Services,1 to 2,1 to 2,0 to 5,11 - 25%,Yes,No,No


## 3.4 Feature Extraction
Extracting a new binary flag, `allows_all_majors`, by parsing the `job_description` column.

In [114]:
all_majors_condition = internship_positions.job_description.str.contains(
    r"semua\sjurusan|jurusan\sapa.*|all\smajors|any\smajor",
    case=False
)

internship_positions["allows_all_majors"] = np.where(all_majors_condition, "Yes", "No")

display(internship_positions.sample(5))

,job_id,published_at,job_title,company,regency_city,education_level,allowed_major,job_description,weekly_working_day,requested_quota,...,acceptance_percentage,job_category,requested_quota_category,approved_quota_category,applicant_count_category,acceptance_percentage_category,allows_bachelor_level,allows_diploma_level,allows_profession_level,allows_all_majors
14742,a23f826d-8497-4ce2-9360-473cf41812fd,2026-07-16T12:35:40+07:00,Asisten Statistisi,BPS Kabupaten Musi Banyuasin,Kab. Musi Banyuasin,"Diploma, Bachelor","Ilmu Ekonomi, Statistika dan Sains Data, Siste...","Membantu pengumpulan, pengolahan, verifikasi, ...",5,2,...,12.50,"Education, Training & Government",1 to 2,1 to 2,11 to 20,11 - 25%,Yes,Yes,No,No
4127,a2415dbc-f1c6-4889-a8e7-adde021d0932,2026-07-16T10:52:06+07:00,Junior RPA Developer - Project,PT Infomedia Nusantara,Kota Adm. Jakarta Selatan,"Diploma, Bachelor, Profession","Teknik Informatika, Teknologi Informasi, Ilmu ...",Mengembangkan proses otomatisasi menggunakan t...,5,1,...,20.00,IT & Data,1 to 2,1 to 2,0 to 5,11 - 25%,Yes,Yes,Yes,No
2511,a2272405-7890-4542-844f-33decd6efec6,2026-07-23T20:46:28+07:00,DESIGN MEDIA,Rumah Sakit Umum Dr. H. Koesnadi Kabupaten Bon...,Kab. Bondowoso,Bachelor,"Teknik Informatika, Promosi Kesehatan","Pendidikan minimal Sarjana, memahami standar p...",5,2,...,28.57,"Media, PR & Creative",1 to 2,1 to 2,6 to 10,26 - 50%,Yes,No,No,No
8217,a24155c2-dbb5-4e78-9a2e-1eae3b1c7e27,2026-07-16T10:13:19+07:00,Wholesale Banking Solution Intern - Operations,Perusahaan Perseroan (Persero) PT. Bank Mandiri,Kota Adm. Jakarta Pusat,Bachelor,"Manajemen, Ilmu Ekonomi, Keuangan Dan Perbanka...",Memberikan dukungan analisa data operasional u...,5,3,...,18.75,Finance & Banking,3 to 10,3 to 10,11 to 20,11 - 25%,Yes,No,No,No
7742,a2415f8b-e195-410d-8bd7-c33f6c90b394,2026-07-16T12:31:24+07:00,PENGELOLA KEGIATAN KERJA,LEMBAGA PEMASYARAKATAN KELAS IIB TONDANO,Kab. Minahasa,Bachelor,Ilmu Pertanian,1. Menyusun rencana dan mengelola kegiatan ker...,5,2,...,18.18,Correctional & Social Services,1 to 2,1 to 2,6 to 10,11 - 25%,Yes,No,No,No


# 4. Schema Finalization
Reorganizing the final 14 columns into a logical analytical structure before exporting.

In [115]:
# Get all the columns
internship_positions.columns

Index(['job_id', 'published_at', 'job_title', 'company', 'regency_city',
       'education_level', 'allowed_major', 'job_description',
       'weekly_working_day', 'requested_quota', 'approved_quota',
       'applicant_count', 'province', 'acceptance_percentage', 'job_category',
       'requested_quota_category', 'approved_quota_category',
       'applicant_count_category', 'acceptance_percentage_category',
       'allows_bachelor_level', 'allows_diploma_level',
       'allows_profession_level', 'allows_all_majors'],
      dtype='str')

In [116]:
# Reorganize the position of the columns
final_cols = [
    "job_id",
    "published_at",
    "job_title",
    "job_category",
    "company",
    "regency_city",
    "province",
    "allowed_major",
    "allows_all_majors",
    "allows_bachelor_level",
    "allows_diploma_level",
    "allows_profession_level",
    "job_description",
    "weekly_working_day",
    "requested_quota_category",
    "approved_quota_category",
    "applicant_count_category",
    "acceptance_percentage_category",
    "requested_quota",
    "approved_quota",
    "applicant_count",
    "acceptance_percentage",
]

internship_postings = internship_positions[final_cols]

display(internship_postings.sample(10))

,job_id,published_at,job_title,job_category,company,regency_city,province,allowed_major,allows_all_majors,allows_bachelor_level,...,job_description,weekly_working_day,requested_quota_category,approved_quota_category,applicant_count_category,acceptance_percentage_category,requested_quota,approved_quota,applicant_count,acceptance_percentage
23393,a2417634-93a6-4d4d-b00e-49c48450fe48,2026-07-16T12:20:54+07:00,PENGELOLA BMN,"HR, Admin & Management",LEMBAGA PEMASYARAKATAN KELAS IIB BLANGKAJEREN,Kab. Gayo Lues,Aceh,Manajemen Aset Publik,No,Yes,...,1. Mengelola aset dan inventaris milik negara ...,6,1 to 2,1 to 2,11 to 20,0 - 10%,1,1,14,6.67
19787,a2410dcc-b4bd-440e-addc-c3759b193ab2,2026-07-16T11:50:22+07:00,PENGELOLA KEGIATAN KERJA,Correctional & Social Services,LEMBAGA PEMASYARAKATAN PEREMPUAN KELAS IIA PAL...,Kota Palembang,Sumatera Selatan,"Budidaya Perikanan, Agribisnis Peternakan, Bud...",No,Yes,...,$28,6,1 to 2,1 to 2,11 to 20,0 - 10%,2,2,20,9.52
26637,a242e7d2-8e21-456f-83ad-936a99ff31be,2026-07-16T11:53:07+07:00,PENGELOLA SDM,"HR, Admin & Management",RUMAH TAHANAN NEGARA KELAS IIB BANJARNEGARA,Kab. Banjarnegara,Jawa Tengah,Manajemen,No,Yes,...,1. Mengumpulkan data dan informasi yang releva...,6,1 to 2,1 to 2,21 to 50,0 - 10%,1,1,21,4.55
4936,a2430057-b00c-4f21-86e5-ca8e97ad75b1,2026-07-16T10:26:27+07:00,Digital Channel,IT & Data,PT Xlsmart Telecom Sejahtera Tbk,Kota Adm. Jakarta Selatan,DKI Jakarta,"Informatika, Ilmu Komputer, Teknik informatika",No,Yes,...,• Mendukung pengembangan dan pemeliharaan kana...,5,3 to 10,3 to 10,21 to 50,11 - 25%,10,10,41,23.81
6939,a241f8c5-c87f-4d73-ba47-45e400fac8f7,2026-07-16T10:40:19+07:00,Building Management Intern,"HR, Admin & Management",PT. Graha Sarana Duta,Kota Adm. Jakarta Pusat,DKI Jakarta,"Administrasi BIsnis, Manajemen, Teknik Industr...",No,Yes,...,Peserta magang akan mendukung fungsi customer ...,5,1 to 2,1 to 2,0 to 5,11 - 25%,1,1,5,16.67
9624,a240e17b-5874-47a6-ba21-0f3192e691e4,2026-07-16T12:34:18+07:00,PENERJEMAH,Language & Translation,KANIM KELAS I KHUSUS NON TPI JAKARTA BARAT,Kota Adm. Jakarta Barat,DKI Jakarta,"Sastra Cina, Sastra Inggris",No,Yes,...,"1. Menerjemahkan dokumen, surat, atau naskah r...",5,1 to 2,1 to 2,6 to 10,11 - 25%,1,1,6,14.29
23352,a2413d4c-89ae-4941-a423-5b7b609a192f,2026-07-16T12:34:57+07:00,Asisten Publisitas dan Kehumasan,"Media, PR & Creative",BPS Kabupaten Takalar,Kab. Takalar,Sulawesi Selatan,"Jurnalistik, Hubungan Masyarakat, Ilmu Komunik...",No,Yes,...,Mendukung kegiatan kehumasan seperti penyusuna...,5,1 to 2,1 to 2,11 to 20,0 - 10%,1,1,14,6.67
22348,a2392287-a374-4cf0-bc0a-8e7396b306b4,2026-07-16T10:44:52+07:00,Magang Perawat,Healthcare & Medical,PT Orbita,Kota Makassar,Sulawesi Selatan,Ilmu Keperawatan,No,Yes,...,membantu melakukan asuhan keperawatan dan mela...,6,3 to 10,3 to 10,21 to 50,0 - 10%,4,4,50,7.84
26186,a2411cb3-6efa-451c-b240-028271a20829,2026-07-16T12:11:18+07:00,Penata Usahaan Persedian dan BMN,"HR, Admin & Management",Balai Besar Laboratorium Kesehatan Masyarakat ...,Kota Makassar,Sulawesi Selatan,"Akuntansi Manajemen, Kesehatan Masyarakat",No,Yes,...,"1. Melaksanakan penerimaan, pemeriksaan, penca...",5,1 to 2,1 to 2,21 to 50,0 - 10%,2,2,38,5.13
18123,a2419511-0f21-471c-9ea1-3fec392140ff,2026-07-16T09:50:11+07:00,Staf Tax Kantor Pusat,Finance & Banking,Nusantara Medika Utama,Kota Mojokerto,Jawa Timur,"Manajemen Perpajakan, Akuntansi Perpajakan, Pa...",No,No,...,Bertanggung jawab atas pengelolaan kewajiban p...,5,1 to 2,1 to 2,6 to 10,0 - 10%,1,1,9,10.00


In [117]:
# Load the final, clean data to a local directory
internship_postings.to_parquet(
    INTERIM_DATA_DIR / "internship_postings.parquet", index=False
) 